## FixBikeMVP

Redo by @anastassiavybornova

In [1]:
# import packages
import pandas as pd
import osmnx as ox
import matplotlib.pyplot as plt
import os

# import functions
from fixbikenet.functions import *

### User Input:

In [2]:
city_name = 'Copenhagen Municipality' # used for Overpass API query
proj_crs = "EPSG:25832"

radius = 2000 # in meters; used to cut-off paths for betweenness centrality compute
maxgap = 50 # in meters; maximum distance between node pairs to be considered as potential gap
penalty = {
    0: 5,
    1: 1
} # factor by which length is multiplied to get weight for NONprotected edges

### Fetch data from OSM

In [3]:
# fetch street network data from osmnx
g = ox.graph_from_place(
    city_name, network_type='all', simplify=False
)
g = ox.simplify_graph(
    g,
    edge_attrs_differ=['highway']
)

In [4]:
# export osmnx data to gdfs
nodes_gdf, edges_gdf = ox.graph_to_gdfs(
    g,
    nodes=True,
    edges=True,
    node_geometry=True,
    fill_edge_geometry=True
)

# project to proj_crs
nodes_gdf = nodes_gdf.to_crs(proj_crs)
edges_gdf = edges_gdf.to_crs(proj_crs)

# edges_gdf.plot()

### Map highway tags to binary classification: protected bikeinfrastructure Yes/No

currently this is done with the simple binary mapping as listed in the yaml file: `config/config_osm.yml`

In [5]:
g = map_edges_to_bike_infrastructure(g)

### Drop parallel edges

g is at this stage a MultiDiGraph. We will now:
* go through all instances of parallel edges
* if they have differing `pbi` values: keep only pbi=1 edge(s) (with bikeinfra)
* if they have equal `pbi` values: *for now*, keep both & let nx take care of it through last step, which is >>
* MultiGraph to Graph conversion (keeps only 1 of each parallel edges, arbitrarily)

In [6]:
edges_to_drop = find_edges_to_drop(g)

In [7]:
# sanity check
print("edges before:", len(g.edges))
print("dropping edges:", len(edges_to_drop))
g.remove_edges_from(edges_to_drop)
print("edges after:", len(g.edges))

edges before: 129438
dropping edges: 159
edges after: 129279


### Convert to undirected simple (not multi) graph

Attention: going from MultiDiGraph to Graph does 2 things:
* drop all parallel edges *randomly*
* remove all directionality (uv==vu)

FR: make this more sensitive (as to *which* of the parallel edges to drop)

In [8]:
# Capital-G: the Graph() object we will be working with from now on
G = nx.Graph(g)

### Add weight attribute to all edges

weight = length times unprotected penalty

In [9]:
G = weigh_edges(G, penalty)

### Create list of all contact nodes

Contact nodes are nodes whose edge set has a pbi set == {0,1} (i.e. that has both protected and unprotected edges)

In [10]:
contact_nodes = find_contact_nodes(G)
print("contact nodes found:", len(contact_nodes))

contact nodes found: 14544


### Create list of all potential gaps

Here defined as: contact node pair combinations **within `maxgap` euclidean distance from each other**

In [11]:
potential_gaps = find_potential_gaps(contact_nodes, nodes_gdf, maxgap)
print("potential gaps found:", len(potential_gaps))

potential gaps found: 142482


### Determine which of the potential gaps are actual gaps by:

1. Running naive shortest path (by length) for all contact nodes
2. Creating an edge list from nodes in shortest paths (convention: `tuple(sorted([u,v]))`)
3. Only saving those gaps that have no bicycle infrastructure edges at all

In [12]:
found_gaps, found_gaps_nsp = find_actual_gaps(G, potential_gaps)

### From list of gaps' nodelists, get all G edges that form part of gaps

In [13]:
edgelist = []
for nodelist in found_gaps_nsp:
    edgelist += [tuple(sorted(z)) for z in zip(nodelist, nodelist[1:])]
# deduplicate
edgelist = list(set(edgelist))

### Weighted shortest paths for entire network (with lambda) to compute local edge betweenness centralities

In [14]:
ebc = compute_local_betweenness_centrality(G, nodes_gdf, radius)

### Rank gaps by B

In [15]:
Bs = rank_gaps_by_b(found_gaps_nsp, G, ebc)

### Order gaps

In [ ]:
df = pd.DataFrame(
    {
        "gap": found_gaps,
        "B": Bs,
        "nodelist": found_gaps_nsp
    }
)
df = df.sort_values(by="B", ascending=False).reset_index(drop=True)
df.head()

In [ ]:
# Sanity check: gap node pairs we're currently finding
fig, ax = plt.subplots(1,1, figsize = (20,20))
nodes_gdf.plot(ax=ax, markersize=0.1, color = "grey")
nodes_gdf.loc[list(set([node for gap in list(df[df.B>7000]["gap"]) for node in gap]))].plot(
    ax=ax,
    markersize=5,
    color="red"
)
ax.set_axis_off()

### Visualize gaps: will be done in separate NB, below saving the data needed for viz

In [ ]:
os.makedirs("./data/", exist_ok=True)
nodes_gdf.to_file("./data/nodes_gdf.gpkg", index=False)
edges_gdf.to_file("./data/edges_gdf.gpkg", index=False)
df.to_json("./data/df.json")